# Field validation — `mixing_parameters` (DEPTH pipeline)

| | |
|---|---|
| Subset | `mixing_parameters` |
| Pipeline | DEPTH |
| Timestep | 2012-11-09 12:00:00 |
| Domain | one 720 × 720 × 51 tile (≈1400 × 1400 km), set in Section 1 |
| Depth levels | `sfc`, `z25m`, `mld`, `mld_mean` |
| Data | computed on the fly from `s3://dbof/LLC4320_RAW/DEPTH/` |
| Plan | `prompts/field_validation_depth.md` |
| Field reference | `docs/Fields.md` |

Rows of the map and PDF figures are **depth levels**, not regions — that
is the one structural difference from the surface notebooks.

**Tier-1/2 sparkle candidates.**  Every field here is a RATIO, so an artifact in a denominator becomes a spike in the output.  `Bu = (Ro/Fr)²` is the worst case: it squares a ratio of two quantities that each carry an artifact.  Section 5b measures the vertical stencil that feeds N², which is under `Fr` and `R_ib`.

## Section 1 — Setup

Everything configurable is in the next cell: the **region**, the date,
the depth levels, and the zoom size.  Change `REGION` to validate a
different part of the ocean — any key in `dbof.plotting.regions.REGIONS`
that carries a `zoom` anchor.

Default is the Gulf Stream, anchored at 60°W / 37°N — dynamically
active in every field this project computes, and the same point the
surface notebooks zoom into, so surface and depth look at the same
water.


In [ ]:
# ---- knobs -------------------------------------------------------------
REGION       = "gulf_stream"          # any region with a 'zoom' anchor
DATE         = "2012-11-09 12:00:00"  # the only DEPTH date transferred so far
LEVELS       = ("sfc", "z25m", "mld", "mld_mean")
ZOOM_HALF_KM = 100.0                  # -> a 200 x 200 km zoom box

SUBSET   = "mixing_parameters"
PIPELINE = "DEPTH"
RAW_VARS = ["Theta", "Salt", "U", "V"]

# Profiles (Figure 3)
N_PROFILES        = 5       # <= 5; the fixed location colours are not cycled
PROFILE_SEED      = 42      # same 5 columns for every field in the notebook
PROFILE_MAX_DEPTH = 500.0   # depth-axis limit, m; None = full 969 m column
# ------------------------------------------------------------------------

import dask
import numpy as np

import dbof.preprocessing.calculate_fields as CF
import dbof.preprocessing.calculate_fields_at_depth as CFAD
from dbof.plotting import depth_figures as dfig
from dbof.plotting.field_cmaps import load_field_cmaps
import dbof.utils.native_gradient as NG
from dbof.preprocessing.vertical_helpers import (
    _interp_w_to_tracer_levels, _vertical_derivative,
)
from dbof.tiles import tile_utils
from dbof.tiles.tile_mapping import rect_ij_to_tile
from dbof.global_dataset_creation.subset_definitions import (
    get_compute_fn, get_subset_definition, expand_channels_with_suffixes,
)

# tile_utils sets the Agg backend when it is imported (it writes QA PNGs
# on headless nodes), so switch back to inline AFTER the dbof imports or
# no figure in this notebook will render.
%matplotlib inline
import matplotlib.pyplot as plt

CMAP_CFG, DIVERGING = load_field_cmaps()

# Channel list straight from the pipeline's own definition -- if the
# subset gains a channel, this notebook picks it up without an edit.
defn = get_subset_definition(PIPELINE, SUBSET)
CHANNELS = expand_channels_with_suffixes(
    defn["compute_features_channels"], list(LEVELS),
    defn.get("extra_channels"),
)
# The mld / mld_mean strategies need MLD, which needs potential
# density.  Loading a subset without Theta/Salt fails deep inside
# mixed_layer_depth with an unhelpful AttributeError, so check here.
if {"mld", "mld_mean"} & set(LEVELS):
    _need = {"Theta", "Salt"} - set(RAW_VARS)
    assert not _need, (
        f"LEVELS includes an MLD-based level, so RAW_VARS must include "
        f"{sorted(_need)} -- MLD is derived from potential density.")

print(f"subset   : {SUBSET}")
print(f"channels : {CHANNELS}")

## Section 2 — Load the tile and compute the fields

We do **not** run `generate-global` here.  That would compute the whole
planet in order to look at one place.

Instead this notebook works on **one tile** — the 720 × 720 × 51 block
the `dbof.tiles` workflow already defines: one LLC face, the full water
column, about 1400 × 1400 km, centred on the region's anchor.  A tile is
*exactly one chunk* of the depth store, so loading it costs one S3 GET
per variable (~106 MB per 3D field).  Tiles are 720-aligned and faces
are 6 × 720 wide, so a tile can never straddle two faces.

Then the **production** compute function for this subset runs on it,
and internally applies the four depth strategies.  Same code as
production, one tile's worth of data.

One thing this costs us: the tile's xgcm grid has **no face
connections**, so cells near the boundary have no neighbours and their
horizontal gradients are wrong.  That rim is NaN'd, using the per-field
widths `tiles/field_registry.py` already records (0 for purely vertical
fields, 1 for staggered interpolation, 3 for gradient and Jacobian
chains).

**A tile samples the region, it does not cover it.**  "Gulf Stream"
here means the ~1400 km tile around 60°W / 37°N — not the whole
80–40°W box the surface notebooks use as a row.


In [ ]:
# Anchor -> rect pixel -> the tile that contains it.
S3 = tile_utils._resolve_s3_source(None)
ANCHOR_LON, ANCHOR_LAT = dfig.region_anchor(REGION)

i_rect, j_rect = tile_utils.latlon_to_rect_ij(ANCHOR_LON, ANCHOR_LAT, S3)
tile = rect_ij_to_tile(i_rect, j_rect)
print(f"region : {REGION} anchored at ({ANCHOR_LON}, {ANCHOR_LAT})")
print(f"tile   : idx {tile.tile_idx}, face {tile.face_idx}, "
      f"j={tile.j_face_slice}, i={tile.i_face_slice}")

# Load the tile + its grid, then merge and build a LOCAL xgcm grid.
ds_grid = tile_utils._load_grid_for_tile(S3, tile)
ds_raw = tile_utils._load_tracers_for_tile(S3, DATE, tile, RAW_VARS)
ds_merge, xgrid = tile_utils._build_tile_context(ds_raw, ds_grid)

XC, YC = dfig.tile_coords(ds_grid)
LAND = dfig.tile_land_mask(ds_grid)
print(f"extent : lon [{XC.min():.2f}, {XC.max():.2f}], "
      f"lat [{YC.min():.2f}, {YC.max():.2f}], "
      f"land {100 * LAND.mean():.1f}%")

### The finals, and the intermediates the figures need

`get_compute_fn("DEPTH", SUBSET)` is the production entry point — the
same callable `generate-global` dispatches to — so the finals below are
the pipeline's own numbers.

The **intermediates** are a different matter: the global products never
store them, so they are recomputed here from the same `ds_merge` the
finals came from.  That is deliberate — it means each figure's chain
shows the actual steps, not a reconstruction.


In [ ]:
store = get_compute_fn(PIPELINE, SUBSET)(ds_merge, xgrid, CHANNELS)
print(f"computed : {sorted(store)}")

mld = CFAD.mixed_layer_depth(ds_merge)

PROFILE_3D = {
    "N2": CFAD.buoyancy_frequency_squared(ds_merge),
    "gradb2": CF.grad_b2(ds_merge, xgrid),
    # The finals, kept in 3D so Figure 3 can profile them.
    "Fr": CFAD.froude_number(ds_merge, xgrid, mld=mld),
    "Ro": CF.rossby_number(ds_merge, xgrid),
    "Bu": CFAD.burger_number(ds_merge, xgrid, mld=mld),
    "R_ib": CFAD.balanced_richardson_number(ds_merge, xgrid),
}
# Fields the production call already reduced to levels: keep them in 3D
# for the profiles, but do not recompute their level slices.
STORE_BASES = {dfig.channel_base(k, LEVELS) for k in store}
live = dfig.compute_levels(
    {k: v for k, v in PROFILE_3D.items() if k not in STORE_BASES},
    ds_merge, mld=mld, levels=LEVELS)

# Both are 2D by nature, so neither has a depth profile.
live["mixed_layer_depth"] = mld
live["coriolis_f"] = CF.coriolis_parameter(ds_merge, xgrid)
live = dict(zip(live, dask.compute(*live.values(), retries=10)))

In [ ]:
# How wide the invalid rim is for this subset, straight from the tile
# registry (0 here: nothing in this chain takes a horizontal gradient).
EDGE_MARGIN = dfig.edge_margin_for(
    list(defn["compute_features_channels"])
    + list(defn.get("extra_channels") or []))

# NaN that rim, mask land with the surface hFacC (what production does),
# and reshape into the {base: {level: (x, y, arr)}} the figures take.
level_arrays = dfig.pack_tile_levels(
    {**live, **store}, XC, YC, edge_margin=EDGE_MARGIN,
    land_mask=LAND, levels=LEVELS)

In [ ]:
# Five ocean columns, seeded and spread across the tile, reused by every
# field in this notebook so the profile panels are comparable.
POINTS = dfig.pick_profile_points(
    LAND, n=N_PROFILES, edge_margin=max(EDGE_MARGIN, 1),
    seed=PROFILE_SEED)

# Full water column at those five points -- a few hundred numbers per
# field, so this is cheap next to the maps.
PROFILES, DEPTH_M = dfig.sample_profiles(PROFILE_3D, ds_merge, POINTS)

# MLD at each point, to mark on the profiles.
MLD_AT_POINTS = (
    [level_arrays["mixed_layer_depth"]["sfc"][2][j, i] for j, i in POINTS]
    if "mixed_layer_depth" in level_arrays else None)

## Section 3 — Subset: `mixing_parameters`

| Channel | Kind |
|---|---|
| `Fr_{sfx}`, `Ro_{sfx}`, `Bu_{sfx}`, `R_ib_{sfx}` | base × depth suffixes |

All four are dimensionless.  None is validated anywhere else, but all
four rest on N² and MLD, which are validated in `stratification.ipynb`.


## Section 4 — Field & dependency table

| FIELD | UNITS | EQUATION | DEPENDS ON | CODE |
|---|---|---|---|---|
| `Fr_{sfx}` | — | Fr = speed / (N·MLD) | U, V, N2, MLD | `calculate_fields_at_depth.froude_number` |
| `Ro_{sfx}` | — | Ro = ζ/f | J, f | `calculate_fields.rossby_number` |
| `Bu_{sfx}` | — | Bu = (Ro/Fr)² | Ro, Fr | `calculate_fields_at_depth.burger_number` |
| `R_ib_{sfx}` | — | R_ib = N²f² / \|∇_h b\|² | N2, f, gradb2 | `calculate_fields_at_depth.balanced_richardson_number` |

**Every field here is a ratio — read the denominators.**

- `Fr` divides by `N·MLD`.  Where the centred vertical stencil cancels
  and N² → 0 spuriously, Fr → ∞.  Note also that `froude_number` uses
  `sqrt(|N²|)`, so a statically unstable column is silently treated as
  stratified.
- `Bu = (Ro/Fr)²` compounds: Ro carries the case-3 ECCO Jacobian
  artifact, Fr carries N²'s vertical artifact, and then the ratio is
  squared.  Expect this to be the noisiest channel in the subset.
- `R_ib` is the best-behaved of the four: its `|∇_h b|²` denominator is
  the square-before-interp form (the surface fix, `docs/Gradients.md`
  case 1), so only its N² numerator is exposed.
- `Fr` also interpolates U and V to centres **without** the CS/SN
  rotation.  For a speed magnitude that is rotation-invariant so the
  number is right, but it is the only velocity path in the codebase
  that skips the rotation.

N² is floored at 0 in `R_ib` (and in `Ri`), so R_ib = 0 is ambiguous
between "unstable" and "unstratified".

**Tile edge rim:** `edge_margin = 3` (Ro and R_ib carry gradient
chains).


## Section 5 — Per-field validation

Three figures per field.

**Figure 1 — maps.**  Columns are the dependency chain, raw → final.
Rows are the four depth levels over the whole tile, then the same four
zoomed to a 200 × 200 km box; the crimson square on the whole-tile rows
is where the zoom is.  One colour scale per column, shared by every row
including the zooms, so nothing changes colour when you look closer.

Fields that **do not vary with depth** get two rows instead of eight —
whole tile and zoom.  `mixed_layer_depth` and `ml_heat_content` are
both integrals over the entire water column, so four identical depth
rows would say nothing.  Their 3D chain inputs are shown at the surface
in those figures, and the title says so.

**Figure 2 — PDFs.**  Same columns; four rows, the whole tile at each
level.  Bins are shared down a column, so reading a column top to
bottom shows how the distribution changes with depth.  The zoom boxes
are deliberately absent — too few cells to make an honest histogram.

**Figure 3 — profiles.**  Five ocean columns, spread across the tile
and fixed by a seed so every field profiles the same water.  The
leftmost panel shows where they are, as numbered colour-coded ×; then
one panel per 3D field in the chain, with the **surface at the top and
depth increasing downward**.  Dashed horizontal lines mark each
location's mixed-layer depth.  The location numbers repeat in the
legend, so the five are distinguishable without relying on colour.

("Grid" in the function names below means the rows × columns array of
panels — not the model's Arakawa C-grid, which is `docs/Grid.md`.)


In [ ]:
# Section 5 helpers: one call per figure, shared by every field.
CHAINS = {
    "Fr": ["N2", "mixed_layer_depth", "Fr"],
    "Ro": ["coriolis_f", "Ro"],
    "Bu": ["Ro", "Fr", "Bu"],
    "R_ib": ["N2", "gradb2", "coriolis_f", "R_ib"],
}
LOG_FIELDS = {"gradb2"}

# Fields with no depth dependence -- integrals over the whole column.
# Their figures collapse to 2 rows (whole tile + zoom) / 1 PDF row.
DEPTH_INVARIANT = {"coriolis_f", "mixed_layer_depth"}


def figure1_maps(field):
    """Figure 1: chain across columns, depth down rows."""
    flat = field in DEPTH_INVARIANT
    note = (" | depth-invariant: 3D inputs shown at the surface"
            if flat else "")
    dfig.depth_map_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        region=REGION,
        levels=("sfc",) if flat else LEVELS,
        row_labels=(("whole tile", f"{2 * ZOOM_HALF_KM:.0f} km zoom")
                    if flat else None),
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        zoom_half_km=ZOOM_HALF_KM,
        suptitle=(f"Figure 1 — {field} | {REGION} tile | columns = "
                  f"dependency chain, rows = depth{note}"),
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: PDFs, chain across columns, depth down rows."""
    flat = field in DEPTH_INVARIANT
    dfig.depth_pdf_grid(
        CHAINS[field], level_arrays, CMAP_CFG,
        levels=("sfc",) if flat else LEVELS,
        row_labels=("whole tile",) if flat else None,
        log10_fields=LOG_FIELDS,
        suptitle=(f"Figure 2 — {field} | {REGION} tile | density; "
                  f"land + rim NaNs dropped; bins shared down each column"),
    )
    plt.show()


def figure3_profiles(field):
    """Figure 3: depth profiles at the five fixed locations."""
    dfig.depth_profile_grid(
        CHAINS[field], PROFILES, DEPTH_M, CMAP_CFG,
        points=POINTS, level_arrays=level_arrays, region=REGION,
        mld_at_points=MLD_AT_POINTS,
        diverging_cmaps=DIVERGING,
        log_scale_channels=LOG_FIELDS,
        max_depth=PROFILE_MAX_DEPTH,
        suptitle=(f"Figure 3 — {field} | {REGION} tile | profiles at "
                  f"{len(POINTS)} locations; surface at top"),
    )
    plt.show()

In [ ]:
# Safety net: every field named in a chain must actually have been
# computed, or the figure call fails deep inside matplotlib.
_missing = sorted({f for c in CHAINS.values() for f in c}
                  - set(level_arrays))
assert not _missing, f"chain fields never computed: {_missing}"
print(f"chains OK : {len(CHAINS)} fields, "
      f"{len({f for c in CHAINS.values() for f in c})} distinct columns")

### Fr — Froude number

**Fr = speed / (N · MLD)**  [—]

Ratio of flow speed to the internal wave speed scale.  Fr > 1 means the flow outruns the stratification.  Watch for isolated extreme values: those are cells where N² cancelled to near zero, not fast water.  Cross-check against the `dz_loss` map in Section 5b.

In [ ]:
figure1_maps("Fr")

In [ ]:
figure2_pdfs("Fr")

In [ ]:
figure3_profiles("Fr")

### Ro — Rossby number

**Ro = ζ/f**  [—]

Same field as `rossby_number` in `kinematic.ipynb`, recomputed here as the subset needs it.  Inherits the case-3 Jacobian artifact through ζ.

In [ ]:
figure1_maps("Ro")

In [ ]:
figure2_pdfs("Ro")

In [ ]:
figure3_profiles("Ro")

### Bu — Burger number

**Bu = (Ro / Fr)²**  [—]

Expected to be the noisiest field in the depth pipeline after `ertel_pv`: a squared ratio of two artifact carriers.  Read it on a log scale mentally — the distribution is very heavy-tailed.

In [ ]:
figure1_maps("Bu")

In [ ]:
figure2_pdfs("Bu")

In [ ]:
figure3_profiles("Bu")

### R_ib — balanced Richardson number

**R_ib = N² f² / |∇_h b|²**  [—]

Frontal stability in thermal-wind balance (Thomas, Tandon & Mahadevan 2013); small values flag symmetric-instability favourable fronts.  The cleanest of the four — its denominator uses the square-first gradient.  R_ib = 0 means N² was floored.

In [ ]:
figure1_maps("R_ib")

In [ ]:
figure2_pdfs("R_ib")

In [ ]:
figure3_profiles("R_ib")

## Section 5b — Vertical stencil check

The surface validation found "sparkles": interpolating a finite
difference across the same axis it was differenced along makes the two
neighbouring slopes (−a and +a at an extremum) average to nearly zero,
and squaring turns that into a speck.  `docs/Gradients.md` has the four
cases and which ones we fixed.

**The vertical has the same problem built into the stencil.**  The
production vertical derivative is centred at interior levels:

    (f[k+1] − f[k−1]) / (z[k+1] − z[k−1])

On even spacing that is *identically* the mean of the two one-sided
slopes either side of level k — the same −a/+a cancellation.  The
difference from the horizontal case is that there is no separate
interpolation step to move: the stencil **is** the interpolation, and
it never reads level k.  A one-level inversion or a sharp step is
invisible to it.

So this section measures it rather than eyeballing it, on potential density (the N² source):

| Column | What |
|---|---|
| `dz_centred` | what the pipeline computes |
| `dz_onesided` | the larger-magnitude one-sided slope straddling level k — what a non-cancelling estimate reports |
| `dz_asym` | `1 − |centred| / |one-sided|` |
| `dz_signflip` | 1 where the two one-sided slopes have OPPOSITE signs |

**Read `dz_signflip`, not `dz_asym`.**  The two measure different
things and only one of them is a defect:

- **Curvature** — same-sign slopes of different magnitude, e.g. the
  base of the mixed layer, where the gradient goes from ~0 above to
  large below.  The centred form returns roughly their mean, which is
  a *correct* second-order estimate at a place where the derivative
  genuinely is not well defined.  This drives `dz_asym` toward 0.5 and
  no further, and it is not an error.
- **Cancellation** — opposite-sign slopes, i.e. level k is a vertical
  extremum.  They partly annihilate, the centred form collapses toward
  zero while both one-sided slopes are large, and *that* is the
  vertical analogue of the horizontal sparkle.  `dz_signflip` marks
  exactly these cells.

On even spacing `dz_asym > 0.5` is reachable only through a sign flip,
so the two agree at that boundary.  A large `dz_asym` at the MLD with
no accompanying sign flips means the stencil is doing its job across a
kink — worth knowing, but not a bug.


In [ ]:
# Centred vs one-sided d/dz, reduced to the same four depth levels.
AB_LABEL = "potential density (the N² source)"
AB = dfig.vertical_stencil_ab(CF.potential_density(ds_merge), ds_merge)
ab_arrays = dfig.pack_tile_levels(
    dfig.compute_levels(AB, ds_merge, mld=mld, levels=LEVELS),
    XC, YC, edge_margin=EDGE_MARGIN, land_mask=LAND, levels=LEVELS,
    verbose=False)

dfig.depth_map_grid(
    ["dz_centred", "dz_onesided", "dz_asym", "dz_signflip"],
    ab_arrays, CMAP_CFG,
    region=REGION, levels=LEVELS, diverging_cmaps=DIVERGING,
    zoom_half_km=ZOOM_HALF_KM,
    suptitle=(f"Figure 4 — vertical stencil A/B on {AB_LABEL}: "
              f"centred vs one-sided d/dz, and the cancellation loss"))
plt.show()

dfig.depth_pdf_grid(
    ["dz_asym", "dz_signflip"], ab_arrays, CMAP_CFG, levels=LEVELS,
    suptitle="Figure 5 — stencil asymmetry and sign flips by level")
plt.show()

# Curvature and cancellation, separately.  The last column is the one
# that matters: it is the fraction of the tile where the centred
# stencil is straddling a vertical extremum.
print(f"{'level':<10}{'median asym':>13}{'asym>0.25':>11}"
      f"{'SIGN FLIP':>11}")
print("-" * 45)
for _lev in LEVELS:
    _a = ab_arrays["dz_asym"][_lev][2]
    _sf = ab_arrays["dz_signflip"][_lev][2]
    if not np.isfinite(_a).any():
        continue
    print(f"{_lev:<10}{np.nanmedian(_a):>13.3f}"
          f"{100 * np.nanmean(_a > 0.25):>10.1f}%"
          f"{100 * np.nanmean(_sf > 0.5):>10.1f}%")
print("")
print("asym  = curvature (benign at a kink); "
      "SIGN FLIP = true cancellation.")

## Section 6 — Literature comparison

**PENDING — nothing to build here yet.**

The comparison figure is chosen *after* the literature figure is, not
before.  Once LH picks a paper figure and drops the PNG into
`../literature_figures/` (naming convention
`{field(s)}_{Citation}_{description}.png`), we decide which of our
panels belongs beside it and add a subsection here — one subsection per
reference, using `dbof.plotting.literature_comparison.side_by_side`.

Leave this section as-is until then.


## Summary — did every channel come out sane?

Coverage and range for each channel at each level, then the physical
checks that are worth failing loudly on.


In [ ]:
# Coverage + range per field per level.
print(f"{'field':<22}{'level':<10}{'finite %':>9}"
      f"{'min':>14}{'max':>14}")
print("-" * 69)
for field in sorted(level_arrays):
    for lev in LEVELS:
        arr = level_arrays[field][lev][2]
        finite = np.isfinite(arr)
        pct = 100.0 * finite.mean()
        lo = np.nanmin(arr) if finite.any() else np.nan
        hi = np.nanmax(arr) if finite.any() else np.nan
        print(f"{field:<22}{lev:<10}{pct:>8.1f}%{lo:>14.4g}{hi:>14.4g}")

In [ ]:
# Physical checks.  These assert -- a red cell here is a real problem.
fr = level_arrays["Fr"]["mld"][2]
bu = level_arrays["Bu"]["mld"][2]
rib = level_arrays["R_ib"]["mld"][2]
ro = level_arrays["Ro"]["sfc"][2]

CHECKS = [
    ("Froude number non-negative",
     np.nanmin(fr) >= 0,
     f"min = {np.nanmin(fr):.3f}"),
    ("Burger number non-negative (it is a square)",
     np.nanmin(bu) >= 0,
     f"min = {np.nanmin(bu):.3e}"),
    ("R_ib non-negative (N2 floored at 0)",
     np.nanmin(rib) >= 0,
     f"min = {np.nanmin(rib):.3e}"),
    ("Rossby number order 1 or below for most of the tile",
     np.nanmean(np.abs(ro) < 1.0) > 0.9,
     f"{100 * np.nanmean(np.abs(ro) < 1.0):.1f}% with |Ro| < 1"),
    ("Bu is heavier-tailed than Ro (the compounding is visible)",
     (np.nanpercentile(bu, 99) / max(np.nanmedian(bu), 1e-30))
     > (np.nanpercentile(np.abs(ro), 99) / max(np.nanmedian(np.abs(ro)),
                                               1e-30)),
     f"Bu p99/median = "
     f"{np.nanpercentile(bu, 99) / max(np.nanmedian(bu), 1e-30):.1f}"),
    ("most of the tile has finite Fr (N*MLD > 0)",
     np.isfinite(fr).mean() > 0.5,
     f"{100 * np.isfinite(fr).mean():.1f}% finite"),
]

failures = []
for name, ok, detail in CHECKS:
    print(f"{'OK  ' if ok else 'FAIL'}  {name}  ({detail})")
    if not ok:
        failures.append(name)
assert not failures, f"physical checks failed: {failures}"
print("\nAll physical checks passed.")

---

### Cross-references

- **N² and MLD**, which every field here divides by —
  `stratification.ipynb`.
- **ζ and the Jacobian** behind Ro — `kinematic.ipynb`.
- **|∇b|²**, the R_ib denominator — `frontal_structure.ipynb`.
- **Ri**, the unbalanced cousin of R_ib — `vertical_shear.ipynb`.
